<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_4-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, glob
from pyspark.sql import types as T
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression

In [2]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

In [3]:
# Leer zip y descomprimir
zip_name = "reddit_technology.zip"
extract_dir = "reddit_extraido"
!unzip -q -o {zip_name} -d {extract_dir}

In [4]:
# Leer parquet
parquets = glob.glob(f"{extract_dir}/**/*.parquet", recursive=True)
ruta_parquet = os.path.commonpath([os.path.dirname(p) for p in parquets])
df = (spark.read
      .option("recursiveFileLookup", "true")
      .parquet(ruta_parquet)
)

# Limpieza (eliminados y nulos)
df = (df
    .filter(~((F.col("author") == "[deleted]") |
              (F.col("body").isin("[deleted]", "[removed]"))))
    .filter(F.col("score").isNotNull())
    .select("created_utc", "author", "score", "body")
)

In [5]:
# Manipulación columnas (seleccionar, renombrar y reordenar)
df = (df
       .withColumnRenamed("author", "autor")
       .withColumnRenamed("body", "mensaje")
       .select("created_utc", "autor", "score", "mensaje")
      )

df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+
|created_utc|        autor|score|                                                                         mensaje|
+-----------+-------------+-----+--------------------------------------------------------------------------------+
| 1432313203|      AbeRego|    3|                                 I read this in the cliché teen Simpson's voice.|
| 1432313210|   hefnetefne|    1|How about a law that says you can sue corporations, instead of proclaiming co...|
| 1432313243|newloginisnew|   23|Please share 100% of your browsing history as a comment reply. If you do not,...|
| 1432313244|       NeonHD|    1|My post wasn't supposed to be a question, it was a link which the original ti...|
| 1432313246|        Zoura|   47|As a former BBV employee I would just like to say, well done! Their crappy bu...|
+-----------+-------------+-----+-----------------------------------------------

In [6]:
# Convertir created_utc a timestamp (segundos a fecha)
df = (df
       .withColumn("created_utc", F.col("created_utc").cast("long"))
       .withColumn("fecha_ts", F.from_unixtime("created_utc").cast("timestamp"))
       .withColumn("score", F.col("score").cast("double"))
       .withColumn("mensaje", F.col("mensaje").cast("string"))
      )

In [7]:
# Features simples
df = (df
       .withColumn("longitud_mensaje", F.length("mensaje"))
       .withColumn("num_palabras", F.size(F.split(F.col("mensaje"), r"\s+")))
       .withColumn("hora", F.hour("fecha_ts"))
      )

In [8]:
# Etiqueta binaria: popular si score >= percentil 90
p90 = df.approxQuantile("score", [0.90], 0.01)[0]
df = df.withColumn("etiqueta", F.when(F.col("score") >= F.lit(p90), 1.0).otherwise(0.0))

In [9]:
# Conjunto de datos final para ML
df_ml = (df
         .select("etiqueta", "mensaje", "longitud_mensaje", "num_palabras", "hora")
         .dropna(subset=["etiqueta", "mensaje", "longitud_mensaje", "num_palabras", "hora"])
        )

In [10]:
df_ml.groupBy("etiqueta").count().show()
print("Percentil 90 (p90) del score =", p90)

+--------+------+
|etiqueta| count|
+--------+------+
|     0.0|160610|
|     1.0| 19468|
+--------+------+

Percentil 90 (p90) del score = 14.0


In [11]:
# Convierte el texto en una lista de palabras (tokens).
tokenizador = RegexTokenizer(
    inputCol="mensaje",
    outputCol="tokens",
    pattern=r"\W+"
)

# Quita palabras muy comunes (the, and, etc,...)
removedor_stopwords = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_limpios"
)

# Convierte tokens en un vector numérico de frecuencias
tf = HashingTF(
    inputCol="tokens_limpios",
    outputCol="tf",
    numFeatures=1 << 18
)

# Baja el peso de palabras comunes y sube el de palabras raras
idf = IDF(
    inputCol="tf",
    outputCol="tfidf"
)

# Une todas tus variables en una sola columna caracteristicas
ensamblador = VectorAssembler(
    inputCols=["tfidf", "longitud_mensaje", "num_palabras", "hora"],
    outputCol="caracteristicas"
)

# Modelo Regresión Logística
modelo_lr = LogisticRegression(
    featuresCol="caracteristicas",
    labelCol="etiqueta",
    maxIter=20
)

# Consolida pasos
pipeline = Pipeline(stages=[
    tokenizador,
    removedor_stopwords,
    tf,
    idf,
    ensamblador,
    modelo_lr
])

In [12]:
# Split
entrenamiento, prueba = df_ml.randomSplit([0.7, 0.3], seed=42)

# Entrenar
modelo = pipeline.fit(entrenamiento)

# Predecir
predicciones = modelo.transform(prueba)

predicciones.select("etiqueta", "prediction", "probability").show(5, truncate=False)

# AUC (binaria)
evaluador_auc = BinaryClassificationEvaluator(
    labelCol="etiqueta",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Accuracy
accuracy = predicciones.filter(F.col("etiqueta") == F.col("prediction")).count() / predicciones.count()

print("Accuracy =", accuracy)

+--------+----------+-------------------------------------------+
|etiqueta|prediction|probability                                |
+--------+----------+-------------------------------------------+
|0.0     |0.0       |[0.9923891924955309,0.007610807504469075]  |
|0.0     |0.0       |[0.9999999999999982,1.7763568394002505E-15]|
|0.0     |1.0       |[0.0019410373363536008,0.9980589626636464] |
|0.0     |0.0       |[0.8093468179681886,0.19065318203181136]   |
|0.0     |0.0       |[0.9996523392052791,3.476607947209276E-4]  |
+--------+----------+-------------------------------------------+
only showing top 5 rows
Accuracy = 0.8243714306904991


In [13]:
matriz_confusion = (
    predicciones
    .select(
        F.col("etiqueta").cast("int").alias("etiqueta"),
        F.col("prediction").cast("int").alias("prediccion")
    )
    .groupBy("etiqueta", "prediccion")
    .count()
    .orderBy("etiqueta", "prediccion")
)

matriz_confusion.show()

+--------+----------+-----+
|etiqueta|prediccion|count|
+--------+----------+-----+
|       0|         0|43727|
|       0|         1| 4429|
|       1|         0| 5043|
|       1|         1|  733|
+--------+----------+-----+



En este trabajo se construyó un modelo de clasificación binaria para predecir si un comentario en Reddit sería popular, definido como pertenecer al percentil 90 del score.

El modelo alcanzó una accuracy de 82.4%. Sin embargo, debido al fuerte desbalance de clases (solo aproximadamente 10% de los comentarios son populares), el modelo mostró baja capacidad para identificar correctamente los comentarios populares, detectando únicamente una pequeña fracción de ellos.

Esto sugiere que la popularidad en Reddit no depende únicamente del contenido textual, sino también de factores contextuales no incluidos en el modelo, como visibilidad del post, reputación del autor o dinámica de la comunidad.